In [1]:
pip install yfinance

StatementMeta(, c285a0e3-6f99-4de6-a5b2-9db1c1f06b43, 3, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 1.1 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 1.2 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [yfinance]5/7 [beautifulsoup4]
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# NB01_Universe
# Builds bronze_universe from Nifty 100 constituents,
# industry-to-bucket mapping, and symbol overrides.
# ============================================================

import re
import pandas as pd
from datetime import datetime

RUN_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
RUN_DATE = datetime.utcnow().date()

def read_csv(path):
    return (spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "false")
            .load(path)
            .toPandas())

def norm_cols(df):
    df.columns = [re.sub(r"[^a-z0-9]+", "_", c.strip().lower()).strip("_")
                  for c in df.columns]
    return df

# ------------------------------------------------------------
# 1. Load inputs
# ------------------------------------------------------------
raw = norm_cols(read_csv("Files/reference/nifty500_raw.csv"))
mapping = norm_cols(read_csv("Files/config/industry_classification.csv"))

try:
    overrides = norm_cols(read_csv("Files/config/symbol_overrides.csv"))
except Exception:
    overrides = pd.DataFrame(columns=["symbol", "bucket", "reason"])

required = {"company_name", "industry", "symbol"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(
        f"Nifty file missing columns: {missing}. "
        f"Found: {list(raw.columns)}"
    )

# ------------------------------------------------------------
# 2. Clean universe
# ------------------------------------------------------------
u = raw.copy()
if "series" in u.columns:
    u = u[u["series"].fillna("").str.strip().str.upper().isin(["EQ", ""])]

u["symbol"] = u["symbol"].astype(str).str.strip().str.upper()
u["company_name"] = u["company_name"].astype(str).str.strip()
u["industry"] = u["industry"].fillna("").astype(str).str.strip()
u = u[u["symbol"].ne("") & u["symbol"].ne("NAN")].drop_duplicates("symbol")

print(f"Cleaned universe: {len(u)} companies")

# ------------------------------------------------------------
# 3. Map industry -> bucket
# ------------------------------------------------------------
mapping["industry"] = mapping["industry"].astype(str).str.strip()
mapping["bucket"] = mapping["bucket"].astype(str).str.strip().str.upper()

u = u.merge(mapping[["industry", "bucket", "reason"]],
            on="industry", how="left")

# Industries not in the mapping file must be reviewed, not silently included
u["bucket"] = u["bucket"].fillna("UNMAPPED")
u["reason"] = u["reason"].fillna("Industry not in industry_classification.csv")
u["classification_source"] = "INDUSTRY_MAP"

# ------------------------------------------------------------
# 4. Apply symbol overrides (take precedence)
# ------------------------------------------------------------
if len(overrides) > 0 and "symbol" in overrides.columns:
    overrides["symbol"] = overrides["symbol"].astype(str).str.strip().str.upper()
    overrides["bucket"] = overrides["bucket"].astype(str).str.strip().str.upper()
    ov = overrides.drop_duplicates("symbol", keep="last").set_index("symbol")

    mask = u["symbol"].isin(ov.index)
    u.loc[mask, "bucket"] = u.loc[mask, "symbol"].map(ov["bucket"])
    u.loc[mask, "reason"] = u.loc[mask, "symbol"].map(ov["reason"])
    u.loc[mask, "classification_source"] = "MANUAL_OVERRIDE"

valid_buckets = {"ELIGIBLE", "UTILITY", "CYCLICAL", "FINANCIAL",
                 "REVIEW", "UNMAPPED"}
bad = set(u["bucket"]) - valid_buckets
if bad:
    raise ValueError(f"Invalid bucket values: {bad}")

# ------------------------------------------------------------
# 5. Derived fields
# ------------------------------------------------------------
u["yahoo_symbol"] = u["symbol"] + ".NS"
u["in_scope"] = u["bucket"].isin(["ELIGIBLE", "UTILITY"])
u["rule_set"] = u["bucket"].map(
    {"ELIGIBLE": "NON_FINANCIAL", "UTILITY": "UTILITY"}
).fillna("NOT_APPLICABLE")
u["universe_date"] = str(RUN_DATE)
u["ingested_at"] = RUN_TS

cols = ["symbol", "yahoo_symbol", "company_name", "industry", "bucket",
        "reason", "classification_source", "in_scope", "rule_set",
        "universe_date", "ingested_at"]
u = u[cols]

# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------
(spark.createDataFrame(u)
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_universe"))

print(f"Saved bronze_universe: {len(u)} companies")
print("\n=== BUCKETS ===")
print(u["bucket"].value_counts().to_string())
print(f"\nIn scope (ELIGIBLE + UTILITY): {int(u['in_scope'].sum())}")

if (u["bucket"] == "UNMAPPED").any():
    print("\nUNMAPPED industries (add them to industry_classification.csv):")
    print(u.loc[u["bucket"] == "UNMAPPED",
                ["symbol", "company_name", "industry"]].to_string(index=False))

StatementMeta(, c285a0e3-6f99-4de6-a5b2-9db1c1f06b43, 4, Finished, Available, Finished, False)

/tmp/ipykernel_6632/1351231326.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
/tmp/ipykernel_6632/1351231326.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_DATE = datetime.utcnow().date()


Cleaned universe: 495 companies
Saved bronze_universe: 495 companies

=== BUCKETS ===
bucket
ELIGIBLE     343
FINANCIAL     98
REVIEW        31
UNMAPPED      23

In scope (ELIGIBLE + UTILITY): 343

UNMAPPED industries (add them to industry_classification.csv):
   symbol                                         company_name          industry
    DMART                               Avenue Supermarts Ltd. Consumer Services
      BLS                      BLS International Services Ltd. Consumer Services
 FIRSTCRY                             Brainbees Solutions Ltd. Consumer Services
 CARTRADE                                   Cartrade Tech Ltd. Consumer Services
  DEVYANI                           Devyani International Ltd. Consumer Services
  EIHOTEL                                             EIH Ltd. Consumer Services
  ETERNAL                                         Eternal Ltd. Consumer Services
    NYKAA                         FSN E-Commerce Ventures Ltd. Consumer Services
ITCHOTELS 

In [3]:
%%sql
SELECT COUNT(*) AS in_scope_companies
FROM bronze_universe
WHERE in_scope = true;

StatementMeta(, c285a0e3-6f99-4de6-a5b2-9db1c1f06b43, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>